# 03 · Geração fundamentada: configurações A e B
- **A** — modelo base **sem contexto** (mesmo contrato JSON, sem trechos): mede o que o modelo "sabe" sozinho e o quanto alucina.
- **B** — **RAG simples**: top-3 trechos + limiar do notebook 02 + validação do JSON (citação obrigatória, só ids fornecidos, 1 nova tentativa) + recusa.

Mesmo modelo, mesmo decodificador (greedy), mesmo gabarito. Resultados em `results/A_base.jsonl` e `results/B_rag.jsonl`.

In [4]:
# --- Ambiente: funciona no Colab (via VSCode ou navegador) e localmente ---------------
import os, sys, subprocess
from pathlib import Path

REPO_URL = ""   # opcional: URL do repositório GitHub, se o projeto ainda não estiver na máquina do Colab

try:
    os.getcwd()
except FileNotFoundError:      # a pasta atual do kernel foi apagada/recriada: volta para um lugar válido
    os.chdir("/content" if Path("/content").exists() else str(Path.home()))

def achar_raiz():
    cwd = Path.cwd()
    cands = [cwd, *cwd.parents, Path("/content/GS2-GENIA"), Path("/content/drive/MyDrive/GS2-GENIA"),
             Path("/content/drive/MyDrive/GEN-IA/GS2-GENIA"), Path("/content/drive/MyDrive/GEN-IA")]
    for c in cands:
        if (c / "src" / "config.py").exists():
            return c
    for base in (Path("/content"), Path("/content/drive/MyDrive")):   # busca 1-2 níveis abaixo
        if base.exists():
            for f in list(base.glob("*/src/config.py")) + list(base.glob("*/*/src/config.py")):
                return f.parent.parent
    return None

ROOT = achar_raiz()
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, "/content/GS2-GENIA"], check=True)
    ROOT = Path("/content/GS2-GENIA")
assert ROOT is not None, "Não achei src/config.py: abra o notebook dentro da pasta GS2-GENIA ou preencha REPO_URL."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

ROOT = /content/drive/MyDrive/GS2-GENIA


In [5]:
%pip install -q -r requirements.txt

In [6]:
import json, pandas as pd
from src import config
from src.ingest import load_chunks
from src.evaluate import load_testset, run_config, aggregate
from src.rag import Generator, answer
from src.retrieval import Retriever, load_threshold

chunks = load_chunks()
by_id = {c["chunk_id"]: c for c in chunks}
retriever = Retriever(chunks)
TH = load_threshold()
print("limiar =", TH)
items = load_testset()          # 40 perguntas (dev + teste)
gen = Generator()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

limiar = 0.8358593756158148


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

... (4 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
rows_A = run_config("A_base", items, lambda q: answer(q, gen, mode="base"))
rows_B = run_config("B_rag", items, lambda q: answer(q, gen, retriever, mode="rag", threshold=TH, chunks_by_id=by_id))
for nome, rows in [("A", rows_A), ("B", rows_B)]:
    for sp in ("dev", "teste"):
        print(nome, sp, aggregate([r for r in rows if r["split"] == sp]))

A_base:   0%|          | 0/40 [00:00<?, ?it/s]

B_rag:   0%|          | 0/40 [00:00<?, ?it/s]

A dev {'n': 20, 'acuracia_%': 30.0, 'acerto_respondiveis_%': 0.0, 'recusa_correta_%': 100.0, 'alucinacao_%': 35.0, 'recusa_indevida_%': 50.0, 'json_valido_%': 100.0, 'citacao_correta_%': 0.0}
A teste {'n': 20, 'acuracia_%': 25.0, 'acerto_respondiveis_%': 0.0, 'recusa_correta_%': 83.3, 'alucinacao_%': 35.0, 'recusa_indevida_%': 57.1, 'json_valido_%': 100.0, 'citacao_correta_%': 0.0}
B dev {'n': 20, 'acuracia_%': 55.0, 'acerto_respondiveis_%': 42.9, 'recusa_correta_%': 83.3, 'alucinacao_%': 35.0, 'recusa_indevida_%': 0.0, 'json_valido_%': 90.0, 'citacao_correta_%': 69.2}
B teste {'n': 20, 'acuracia_%': 35.0, 'acerto_respondiveis_%': 35.7, 'recusa_correta_%': 33.3, 'alucinacao_%': 55.0, 'recusa_indevida_%': 0.0, 'json_valido_%': 90.0, 'citacao_correta_%': 64.3}


## Exemplos (para o relatório): resposta com citação, recusa pelo limiar, recusa/alucinação do modelo base

In [8]:
def mostra(q):
    o = answer(q, gen, retriever, mode="rag", threshold=TH, chunks_by_id=by_id)
    print(json.dumps({k: o[k] for k in ("status", "resposta", "gated", "top_score", "avisos")}, ensure_ascii=False, indent=1))
    for c in o.get("citacoes", []):
        print("  fonte:", c["doc_id"], "p.", c["paginas"], "|", c["trecho"][:160].replace("\n", " "), "...")
mostra("Qual é a amplitude de controle recomendada para um supervisor no SCI?")
print("-" * 60)
mostra("Qual foi a taxa Selic definida pelo Banco Central do Brasil em novembro de 2026?")
print("-" * 60)
print("A (sem contexto):", answer("Qual é a amplitude de controle recomendada para um supervisor no SCI?", gen, mode="base")["resposta"])

{
 "status": "ok",
 "resposta": "A amplitude de controle recomendada para um supervisor no SCI varia de três a sete subordinados.",
 "gated": false,
 "top_score": 0.9152045249938965,
 "avisos": []
}
  fonte: DOC007 p. [37, 37] | e para um gerenciamento eficaz de incidentes. Manter a  amplitude de controle é importante porque a segurança e a responsabilização são  prioritárias. ● Dentro  ...
  fonte: DOC008 p. [32, 33] | Excepcionalmente pode ser responsável por um número inferior a três e maior que  sete.  Figura 12 – Amplitude de Controle.        Fonte: Autor, (2023) 33    Val ...
------------------------------------------------------------
{
 "status": "nao_consta",
 "resposta": "Não consta nos documentos da base de conhecimento.",
 "gated": true,
 "top_score": 0.805904746055603,
 "avisos": []
}
------------------------------------------------------------
A (sem contexto): Não consta na legislação específica do Sistema de Controle Integrado (SCI) a recomendação exata da amplitude de 

In [9]:
import gc, torch
del gen; gc.collect(); torch.cuda.empty_cache()